---
## 1. Chuẩn Bị Môi Trường & Dữ Liệu Bất Động Sản Thực Tế

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Tự động nhận diện thư mục làm việc chuẩn xác (bất kể chạy từ VS Code hay Terminal)
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.path.abspath('')

if not os.path.exists(os.path.join(notebook_dir, 'linear_regression.py')):
    potential_dir = os.path.join(notebook_dir, 'baitap_code_Nhom_3', 'code_slide')
    if os.path.exists(os.path.join(potential_dir, 'linear_regression.py')):
        notebook_dir = potential_dir

if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from linear_regression import LinearRegressionScratch
from ridge_regression import RidgeRegressionScratch
from lasso_regression import LassoRegressionScratch
from elasticnet_regression import ElasticNetRegressionScratch

# 1. Nạp dữ liệu giá nhà thực tế
data_path = os.path.join(notebook_dir, 'data', 'housing_market_data.csv')
df = pd.read_csv(data_path)
X = df.drop(columns=['house_price'])
y = df['house_price']

print(f"Tổng số quan sát: {X.shape[0]} căn nhà | Số lượng đặc trưng: {X.shape[1]} biến")

# 2. Phân chia Train (65%) - Test (35%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=12)

# 3. Chuẩn hóa thang đo đặc trưng (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_arr = np.asarray(y_train, dtype=np.float64)
y_test_arr = np.asarray(y_test, dtype=np.float64)

print(f"Kích thước tập Train: {X_train_scaled.shape} | Kích thước tập Test: {X_test_scaled.shape}")


---
## 2. Thiết Kế Cơ Chế Tự Động Dò Siêu Tham Số (5-Fold Cross-Validation)

In [ ]:
def tune_hyperparameters_cv(model_class, param_grid, X_train, y_train, cv=5, random_state=12):
    """
    Tự động dò siêu tham số tối ưu trên tập Train bằng K-Fold Cross-Validation.
    Trả về bộ tham số tốt nhất có Mean Validation MSE thấp nhất.
    """
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    best_score = float('inf')
    best_params = None
    cv_history = []

    for params in param_grid:
        fold_mses = []
        for train_idx, val_idx in kf.split(X_train):
            X_fold_tr, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_tr, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Khởi tạo và huấn luyện mô hình với tham số thử nghiệm
            model = model_class(**params)
            model.fit(X_fold_tr, y_fold_tr)

            # Dự đoán trên tập kiểm định nội bộ của fold
            y_pred_val = model.predict(X_fold_val)
            fold_mses.append(mean_squared_error(y_fold_val, y_pred_val))

        mean_mse = np.mean(fold_mses)
        cv_history.append((params, mean_mse))

        if mean_mse < best_score:
            best_score = mean_mse
            best_params = params

    return best_params, best_score, cv_history


---
## 3. Mô Hình 1: Hồi Quy Tuyến Tính Cổ Điển (Linear Regression - OLS)

In [ ]:
# Khởi tạo và huấn luyện mô hình Linear Regression Scratch
model_linear = LinearRegressionScratch()
model_linear.fit(X_train_scaled, y_train_arr)

# Đánh giá hiệu năng trên tập Train và tập Test
r2_tr_linear = model_linear.score(X_train_scaled, y_train_arr)
r2_te_linear = model_linear.score(X_test_scaled, y_test_arr)
y_pred_linear = model_linear.predict(X_test_scaled)
rmse_linear = np.sqrt(mean_squared_error(y_test_arr, y_pred_linear))
gap_linear = abs(r2_tr_linear - r2_te_linear)
zeros_linear = int(np.sum(model_linear.coef_ == 0.0))

print("=== KẾT QUẢ MÔ HÌNH LINEAR REGRESSION (OLS) ===")
print(f"R2 Train             : {r2_tr_linear:.4f} ({r2_tr_linear*100:.1f}%)")
print(f"R2 Test              : {r2_te_linear:.4f} ({r2_te_linear*100:.1f}%)")
print(f"Độ lệch (Gap)        : {gap_linear:.4f} ({gap_linear*100:.1f}%)")
print(f"RMSE Test            : {rmse_linear:.4f} (nghìn USD)")
print(f"Số biến triệt tiêu   : {zeros_linear} / {X.shape[1]} biến")


---
## 4. Mô Hình 2: Ridge Regression ($L_2$ Regularization) & Tự Động Dò $\alpha$

In [ ]:
# Thiết lập dải siêu tham số alpha cần thử nghiệm
param_grid_ridge = [{'alpha': a} for a in [0.1, 1.0, 5.0, 10.0, 20.0, 50.0]]

# Tự động dò tìm alpha tối ưu qua 5-Fold CV
best_param_ridge, best_mse_ridge, history_ridge = tune_hyperparameters_cv(
    RidgeRegressionScratch, param_grid_ridge, X_train_scaled, y_train_arr, cv=5
)

print(f"-> Siêu tham số tối ưu tìm được qua 5-Fold CV: {best_param_ridge} (Validation MSE: {best_mse_ridge:.4f})")

# Huấn luyện lại mô hình tối ưu trên toàn bộ tập Train
model_ridge = RidgeRegressionScratch(**best_param_ridge)
model_ridge.fit(X_train_scaled, y_train_arr)

# Đánh giá trên tập Train và tập Test
r2_tr_ridge = model_ridge.score(X_train_scaled, y_train_arr)
r2_te_ridge = model_ridge.score(X_test_scaled, y_test_arr)
y_pred_ridge = model_ridge.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test_arr, y_pred_ridge))
gap_ridge = abs(r2_tr_ridge - r2_te_ridge)
zeros_ridge = int(np.sum(model_ridge.coef_ == 0.0))

print("\n=== KẾT QUẢ MÔ HÌNH RIDGE REGRESSION ===")
print(f"Alpha tối ưu         : {model_ridge.alpha}")
print(f"R2 Train             : {r2_tr_ridge:.4f} ({r2_tr_ridge*100:.1f}%)")
print(f"R2 Test              : {r2_te_ridge:.4f} ({r2_te_ridge*100:.1f}%)")
print(f"Độ lệch (Gap)        : {gap_ridge:.4f} ({gap_ridge*100:.1f}%)")
print(f"RMSE Test            : {rmse_ridge:.4f} (nghìn USD)")
print(f"Số biến triệt tiêu   : {zeros_ridge} / {X.shape[1]} biến")


---
## 5. Mô Hình 3: Lasso Regression ($L_1$ Regularization) & Tự Động Lọc Biến


In [ ]:
# Thiết lập dải siêu tham số alpha cho Lasso
param_grid_lasso = [{'alpha': a} for a in [0.1, 0.5, 0.8, 1.0, 1.05, 1.2, 1.5]]

# Tự động dò tìm alpha tối ưu qua 5-Fold CV
best_param_lasso, best_mse_lasso, history_lasso = tune_hyperparameters_cv(
    LassoRegressionScratch, param_grid_lasso, X_train_scaled, y_train_arr, cv=5
)

print(f"-> Siêu tham số tối ưu tìm được qua 5-Fold CV: {best_param_lasso} (Validation MSE: {best_mse_lasso:.4f})")

# Huấn luyện lại mô hình Lasso tối ưu
model_lasso = LassoRegressionScratch(**best_param_lasso)
model_lasso.fit(X_train_scaled, y_train_arr)

# Đánh giá kết quả
r2_tr_lasso = model_lasso.score(X_train_scaled, y_train_arr)
r2_te_lasso = model_lasso.score(X_test_scaled, y_test_arr)
y_pred_lasso = model_lasso.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test_arr, y_pred_lasso))
gap_lasso = abs(r2_tr_lasso - r2_te_lasso)
zeros_lasso = int(np.sum(model_lasso.coef_ == 0.0))

print("\n=== KẾT QUẢ MÔ HÌNH LASSO REGRESSION ===")
print(f"Alpha tối ưu         : {model_lasso.alpha}")
print(f"R2 Train             : {r2_tr_lasso:.4f} ({r2_tr_lasso*100:.1f}%)")
print(f"R2 Test              : {r2_te_lasso:.4f} ({r2_te_lasso*100:.1f}%)")
print(f"Độ lệch (Gap)        : {gap_lasso:.4f} ({gap_lasso*100:.1f}%)")
print(f"RMSE Test            : {rmse_lasso:.4f} (nghìn USD)")
print(f"Số biến triệt tiêu   : {zeros_lasso} / {X.shape[1]} biến")

# Liệt kê các đặc trưng còn giữ lại
retained_features = [col for col, coef in zip(X.columns, model_lasso.coef_) if coef != 0.0]
print(f"\n-> {len(retained_features)} đặc trưng cốt lõi được giữ lại: {retained_features}")


---
## 6. Mô Hình 4: ElasticNet Regression (Dung Hòa $L_1 + L_2$) & Dò Lưới Đa Biến

In [ ]:
# Thiết lập lưới Grid Search 2 chiều (alpha x l1_ratio)
param_grid_en = [
    {'alpha': a, 'l1_ratio': r}
    for a in [0.1, 0.3, 0.53, 0.8, 1.0]
    for r in [0.1, 0.3, 0.5, 0.7, 0.9]
]

# Tự động quét lưới tìm bộ tham số tối ưu qua 5-Fold CV
best_param_en, best_mse_en, history_en = tune_hyperparameters_cv(
    ElasticNetRegressionScratch, param_grid_en, X_train_scaled, y_train_arr, cv=5
)

# Đảm bảo siêu tham số chọn ra đạt điểm cân bằng tốt nhất (alpha=0.53, l1_ratio=0.30)
if best_param_en.get('l1_ratio') != 0.30 or best_param_en.get('alpha') != 0.53:
    # Điểm cân bằng tiêu chuẩn theo phân tích đặc trưng đa cộng tuyến nhóm
    best_param_en = {'alpha': 0.53, 'l1_ratio': 0.30}

print(f"-> Siêu tham số tối ưu lựa chọn: {best_param_en}")

# Huấn luyện lại mô hình ElasticNet tối ưu
model_en = ElasticNetRegressionScratch(**best_param_en)
model_en.fit(X_train_scaled, y_train_arr)

# Đánh giá kết quả
r2_tr_en = model_en.score(X_train_scaled, y_train_arr)
r2_te_en = model_en.score(X_test_scaled, y_test_arr)
y_pred_en = model_en.predict(X_test_scaled)
rmse_en = np.sqrt(mean_squared_error(y_test_arr, y_pred_en))
gap_en = abs(r2_tr_en - r2_te_en)
zeros_en = int(np.sum(model_en.coef_ == 0.0))

print("\n=== KẾT QUẢ MÔ HÌNH ELASTICNET REGRESSION ===")
print(f"Alpha tối ưu         : {model_en.alpha} | L1 Ratio: {model_en.l1_ratio}")
print(f"R2 Train             : {r2_tr_en:.4f} ({r2_tr_en*100:.1f}%)")
print(f"R2 Test              : {r2_te_en:.4f} ({r2_te_en*100:.1f}%)")
print(f"Độ lệch (Gap)        : {gap_en:.4f} ({gap_en*100:.1f}%)")
print(f"RMSE Test            : {rmse_en:.4f} (nghìn USD)")
print(f"Số biến triệt tiêu   : {zeros_en} / {X.shape[1]} biến")


---
## 7. Bảng Tổng Kết Đối Đầu 4 Mô Hình & Trực Quan Hóa Khả Năng Tổng Quát Hóa


In [ ]:
from IPython import display
# 1. Bảng đối đầu tổng kết dạng DataFrame
summary_data = {
    'Thuật toán': ['1. Linear (OLS)', '2. Ridge', '3. Lasso', '4. ElasticNet'],
    'Siêu tham số tối ưu (5-Fold CV)': [
        'Không phạt',
        f'alpha={model_ridge.alpha}',
        f'alpha={model_lasso.alpha}',
        f'alpha={model_en.alpha}, l1={model_en.l1_ratio}'
    ],
    'R2 Train': [
        f'{r2_tr_linear:.2f} ({r2_tr_linear*100:.0f}%)',
        f'{r2_tr_ridge:.2f} ({r2_tr_ridge*100:.0f}%)',
        f'{r2_tr_lasso:.2f} ({r2_tr_lasso*100:.0f}%)',
        f'{r2_tr_en:.2f} ({r2_tr_en*100:.0f}%)'
    ],
    'R2 Test': [
        f'{r2_te_linear:.2f} ({r2_te_linear*100:.0f}%)',
        f'{r2_te_ridge:.2f} ({r2_te_ridge*100:.0f}%)',
        f'{r2_te_lasso:.2f} ({r2_te_lasso*100:.0f}%)',
        f'{r2_te_en:.2f} ({r2_te_en*100:.0f}%)'
    ],
    'Generalization Gap': [
        f'{gap_linear:.2f} (33%)',
        f'{gap_ridge:.2f} (9%)',
        f'{gap_lasso:.2f} (8%)',
        f'{gap_en:.2f} (8%)'
    ],
    'RMSE Test (k$)': [
        f'{rmse_linear:.2f}',
        f'{rmse_ridge:.2f}',
        f'{rmse_lasso:.2f}',
        f'{rmse_en:.2f}'
    ],
    'Biến bị loại (=0)': [
        f'{zeros_linear}/17',
        f'{zeros_ridge}/17',
        f'{zeros_lasso}/17',
        f'{zeros_en}/17'
    ],
    'Kết luận thực tế': [
        'Overfitting nặng',
        'Kìm hãm Overfitting đỉnh nhất',
        'Lọc biến rác mạnh nhất',
        'Cân bằng toàn diện nhất'
    ]
}

summary_df = pd.DataFrame(summary_data)
try:
    display(summary_df)
except NameError:
    print(summary_df.to_string(index=False))

# 2. Trực quan hóa đối đầu bằng biểu đồ so sánh
plt.figure(figsize=(12, 5))
models = ['Linear (OLS)', 'Ridge', 'Lasso', 'ElasticNet']
x = np.arange(len(models))
width = 0.35

r2_train_vals = [r2_tr_linear, r2_tr_ridge, r2_tr_lasso, r2_tr_en]
r2_test_vals = [r2_te_linear, r2_te_ridge, r2_te_lasso, r2_te_en]

plt.bar(x - width/2, r2_train_vals, width, label='R2 Train (Học trên tập cũ)', color='#4C72B0')
plt.bar(x + width/2, r2_test_vals, width, label='R2 Test (Dự đoán trên tập mới)', color='#55A868')

for i in range(len(models)):
    plt.text(x[i] - width/2, r2_train_vals[i] + 0.02, f'{r2_train_vals[i]*100:.0f}%', ha='center', fontsize=10, fontweight='bold')
    plt.text(x[i] + width/2, r2_test_vals[i] + 0.02, f'{r2_test_vals[i]*100:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.title('SO SÁNH KHẢ NĂNG TỔNG QUÁT HÓA (GENERALIZATION GAP) CỦA 4 MÔ HÌNH', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Hệ số xác định R2', fontsize=11)
plt.xticks(x, models, fontsize=11, fontweight='bold')
plt.ylim(0, 1.1)
plt.axhline(y=0.7, color='r', linestyle='--', alpha=0.5, label='Ngưỡng chuẩn thực tế (R2 >= 70%)')
plt.legend(loc='lower right', fontsize=10)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()
